# Neural Network Risk Pipeline

This notebook mirrors the leakage-reduced neural network setup from `ml/src/neural_network_model.py`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "ml" / "data" / "processed" / "cleaned_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.shape

## Leakage Note

The current target is still derived from `tax_ratio`, so the dataset itself has label leakage. The neural network input list below removes `tax_ratio`, raw `tax`, `price_usd`, value-per-price features, and `tax_paid_share` so the model cannot trivially reconstruct the target from direct tax ratio signals. A fully leakage-free model requires redefining the target.

In [ ]:
FEATURE_COLUMNS = [
    "weight_kg",
    "length_m",
    "width_m",
    "height_m",
    "volume_m3",
    "max_dimension_m",
    "dimension_sum_m",
    "density_kg_m3",
    "expected_tax_rate",
    "tax_gap",
    "tax_per_kg",
    "tax_per_m3",
    "log_tax",
]

TARGET_COLUMN = "risk"
removed_leakage_features = [
    "tax_ratio",
    "tax",
    "price_usd",
    "value_per_kg",
    "value_per_m3",
    "tax_paid_share",
]

df[FEATURE_COLUMNS + [TARGET_COLUMN]].head()

In [ ]:
missing_columns = set(FEATURE_COLUMNS + [TARGET_COLUMN]).difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().sum().sort_values(ascending=False).head()

## Clean Split

Fit preprocessing on training data only. Use validation for model selection and early stopping. Keep test untouched until final evaluation.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].map({"LOW RISK": 0, "HIGH RISK": 1})

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

validation_fraction = VALIDATION_SIZE / (1 - TEST_SIZE)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=validation_fraction,
    random_state=RANDOM_STATE,
    stratify=y_train_valid,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

{
    "train_rows": len(X_train),
    "validation_rows": len(X_valid),
    "test_rows": len(X_test),
    "features": FEATURE_COLUMNS,
}

Training and artifact saving live in `ml/src/neural_network_model.py`. Run that script to train candidate architectures on the training split, select using validation metrics, and report final metrics on the held-out test split only.